# Step 0 - Run all (end-to-end orchestrator)

**CSC3119 Assignment 4.** One cell runs the six stage notebooks in order and stops at the first failure, so you never discover a broken step after paying for RunPod time. Run this on the pod:

```bash
jupyter nbconvert --to notebook --execute Step0_Run_All.ipynb \
  --ExecutePreprocessor.timeout=3600 --output Step0_executed.ipynb
```

Or run the plain-Python equivalent from the project root: `python run_all.py`.

In [1]:
from pathlib import Path
import subprocess, sys, time, json

PROJECT = Path.cwd()
while not (PROJECT / "src" / "aq_common.py").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
NB_DIR = PROJECT / "notebooks"

STEPS = [
    "Step1_Data_Preparation.ipynb",
    "Step2_EDA.ipynb",
    "Step3_Modeling.ipynb",
    "Step4_Forecasting.ipynb",
    "Step5_Monitoring.ipynb",
    "Step6_Deployment.ipynb",
]
print(f"Project : {PROJECT}")
print(f"Python  : {sys.executable}")
print(f"Steps   : {len(STEPS)}")

timings = {}
for name in STEPS:
    nb = NB_DIR / name
    print(f"\n>>> running {name} ...", flush=True)
    t0 = time.time()
    proc = subprocess.run(
        [sys.executable, "-m", "nbconvert", "--to", "notebook", "--execute",
         "--inplace", "--ExecutePreprocessor.timeout=3600", str(nb)],
        capture_output=True, text=True)
    dt = time.time() - t0
    timings[name] = round(dt, 1)
    ok = proc.returncode == 0
    print(f"    {'OK ' if ok else 'FAIL'}  {dt:6.1f} s")
    if not ok:
        print(proc.stderr[-1500:])
        raise SystemExit(f"{name} failed - stopping so you can fix it before RunPod.")

print("\nAll stages completed.")
for k, v in timings.items():
    print(f"  {k:34s} {v:7.1f} s")
(PROJECT / "outputs" / "runs" / "run_all_timings.json").write_text(
    json.dumps(timings, indent=2), encoding="utf-8")
print("\noutputs/metrics.json and outputs/runs/pipeline_log.txt are ready to screenshot.")

Project : C:\Users\hp\OneDrive\Desktop\3rd YEAR 1st SEM\AI Model Development\Assignment 4
Python  : C:\Python313\python.exe
Steps   : 6

>>> running Step1_Data_Preparation.ipynb ...


    OK     33.2 s

>>> running Step2_EDA.ipynb ...


    OK     24.4 s

>>> running Step3_Modeling.ipynb ...


    OK    614.4 s

>>> running Step4_Forecasting.ipynb ...


    OK     34.5 s

>>> running Step5_Monitoring.ipynb ...


    OK     59.8 s

>>> running Step6_Deployment.ipynb ...


    OK     19.0 s

All stages completed.
  Step1_Data_Preparation.ipynb          33.2 s
  Step2_EDA.ipynb                       24.4 s
  Step3_Modeling.ipynb                 614.4 s
  Step4_Forecasting.ipynb               34.5 s
  Step5_Monitoring.ipynb                59.8 s
  Step6_Deployment.ipynb                19.0 s

outputs/metrics.json and outputs/runs/pipeline_log.txt are ready to screenshot.
